# **Prepare Dataset**

In [ ]:
import pandas as pd
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from imblearn.over_sampling import SMOTE

In [ ]:
import pandas as pd

df = pd.read_csv('/content/combined_data.csv')
display(df.head())

,domain_name,unified_sentiment,Sentences_clean,Aspects_clean,Aspects_sentiments_list,Aspects_list
0,Attractions,1.0,نصحوني بتجربه حمام الكبريت يمكنكم الدخول مع مج...,الحمام، العاملات، الانجليزيه,"[1, 1, -1]","['الحمام', 'العاملات', 'الانجليزيه']"
1,Attractions,1.0,قلعه ساحره منظر خلاب للمدينه من اعلي القلعه يو...,قلعه، منظر، حديقه نباتات، مجري مائي,"[1, 1, 1, 1]","['قلعه', 'منظر', 'حديقه نباتات', 'مجري مائي']"
2,Attractions,1.0,تبليسي جورجيا من اجمل المدن التي زرتها في حيات...,تبليسي جورجيا، شعبها، الحياه,"[1, 1, 1]","['تبليسي جورجيا', 'شعبها', 'الحياه']"
3,Attractions,1.0,جوله علي المدينه القديمه تبليسي شاردن ممتعه ال...,المدينه القديمه تبليسي، الجوله، جلساتها، تلفري...,"[1, 1, 1, 1, 1, 1]","['المدينه القديمه تبليسي', 'الجوله', 'جلساتها'..."
4,Attractions,1.0,احلي اجازه لمحبي الطبيعه المناظر الخلابه الطبي...,اجازه، المناظر، الاجواء، ميزه,"[1, 1, 1, 0]","['اجازه', 'المناظر', 'الاجواء', 'ميزه']"


In [ ]:
import pandas as pd

df = df[["Sentences_clean", "unified_sentiment"]]
df.dropna(inplace=True)
df.columns = ["text", "label"]
df["label"] = df["label"].astype(int)

print(df.head())
print(df["label"].value_counts())

                                                text  label
0  نصحوني بتجربه حمام الكبريت يمكنكم الدخول مع مج...      1
1  قلعه ساحره منظر خلاب للمدينه من اعلي القلعه يو...      1
2  تبليسي جورجيا من اجمل المدن التي زرتها في حيات...      1
3  جوله علي المدينه القديمه تبليسي شاردن ممتعه ال...      1
4  احلي اجازه لمحبي الطبيعه المناظر الخلابه الطبي...      1
label
 1    4940
-1     715
 0     405
Name: count, dtype: int64


/tmp/ipykernel_3335/261373503.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)
/tmp/ipykernel_3335/261373503.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = df["label"].astype(int)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# **TFIDF+LR**

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(
    X_train_tfidf,
    y_train
)

print("After Oversampling:")
print(pd.Series(y_resampled).value_counts())


After Oversampling:
label
-1    3952
 1    3952
 0    3952
Name: count, dtype: int64


In [ ]:
lr_model = LogisticRegression(
    max_iter=2000
)

lr_model.fit(X_resampled, y_resampled)
preds = lr_model.predict(X_test_tfidf)

print("Accuracy:")
print(accuracy_score(y_test, preds))

print("\nClassification Report:")
print(classification_report(y_test, preds))


Accuracy:
0.8135313531353136

Classification Report:
              precision    recall  f1-score   support

          -1       0.53      0.60      0.56       143
           0       0.29      0.35      0.32        81
           1       0.91      0.88      0.90       988

    accuracy                           0.81      1212
   macro avg       0.58      0.61      0.59      1212
weighted avg       0.83      0.81      0.82      1212



# **Arabert Fine Tuned**

In [ ]:
import pandas as pd
import numpy as np
import re
import torch

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
label_map = {
    -1: 0,
     0: 1,
     1: 2
}

df["label"] = df["label"].map(label_map)

/tmp/ipykernel_3335/3138223275.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = df["label"].map(label_map)


In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LEN = 128

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/4848 [00:00<?, ? examples/s]

Map:   0%|          | 0/1212 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:")
print(class_weights)


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Class Weights:
tensor([2.8252, 4.9877, 0.4089])


In [ ]:
class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs.get("input_ids"),
            attention_mask=inputs.get("attention_mask")
        )
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./arabert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(
    test_df["label"],
    preds
))
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(
    test_df["label"],
    preds
))

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(
    test_df["label"],
    preds
))


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.936835,0.783925,0.880363
2,0.830129,0.896295,0.871287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

              precision    recall  f1-score   support

           0       0.72      0.78      0.75       143
           1       0.24      0.10      0.14        81
           2       0.93      0.96      0.94       988

    accuracy                           0.88      1212
   macro avg       0.63      0.61      0.61      1212
weighted avg       0.86      0.88      0.87      1212



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.72      0.78      0.75       143
           1       0.24      0.10      0.14        81
           2       0.93      0.96      0.94       988

    accuracy                           0.88      1212
   macro avg       0.63      0.61      0.61      1212
weighted avg       0.86      0.88      0.87      1212



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.72      0.78      0.75       143
           1       0.24      0.10      0.14        81
           2       0.93      0.96      0.94       988

    accuracy                           0.88      1212
   macro avg       0.63      0.61      0.61      1212
weighted avg       0.86      0.88      0.87      1212



In [ ]:
trainer.save_model("arabert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]